In [1]:
from sklearn import datasets
import pandas as pd
from sklearn.preprocessing import scale
from sklearn import decomposition

In [2]:
fires = pd.read_csv('fire_climate_dataset.csv')
fires = pd.DataFrame(fires)

In [3]:
X = fires[['fireyear','lat','lon','month','temp_month','precip_month']]
Y = fires['gisacres']

In [4]:
X.shape

(947, 6)

In [5]:
Y.shape

(947,)

In [6]:
X = scale(X)

In [7]:
pca = decomposition.PCA(n_components = 3)
pca.fit(X)

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",3
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD 

In [8]:
scores = pca.transform(X)

In [9]:
scores_df = pd.DataFrame(scores, columns = ['PC1','PC2','PC3'])
scores_df

,PC1,PC2,PC3
0,-1.595522,-0.505863,-0.899494
1,-0.407461,0.134044,0.640724
2,-1.392492,0.783026,0.087512
3,-0.813227,0.798505,0.925992
4,-0.375459,0.147826,0.628460
...,...,...,...
942,0.300113,1.064263,0.801278
943,-1.244659,-1.091326,-0.627017
944,-1.236559,-0.792234,-0.647979
945,1.108572,-3.833269,0.705671


In [10]:
Y.describe()

count       947.000000
mean       1895.349899
std        8519.749726
min           0.030140
25%          17.890754
50%          75.255779
75%         454.380382
max      137894.518963
Name: gisacres, dtype: float64

In [11]:
type(Y)

pandas.Series

In [12]:
Y = pd.to_numeric(Y, errors="coerce")

In [13]:
Y_label = []

for i in Y:
    if i <18:
        Y_label.append('very small')
    elif i < 75:
        Y_label.append('small')
    elif i < 454:
        Y_label.append('large')
    else:
        Y_label.append('very large')

Fires = pd.DataFrame(Y_label, columns = ['Fire Size'])

In [14]:
df_scores = pd.concat([scores_df, Fires], axis=1)
df_scores

,PC1,PC2,PC3,Fire Size
0,-1.595522,-0.505863,-0.899494,very small
1,-0.407461,0.134044,0.640724,very small
2,-1.392492,0.783026,0.087512,very small
3,-0.813227,0.798505,0.925992,very small
4,-0.375459,0.147826,0.628460,small
...,...,...,...,...
942,0.300113,1.064263,0.801278,very small
943,-1.244659,-1.091326,-0.627017,small
944,-1.236559,-0.792234,-0.647979,very large
945,1.108572,-3.833269,0.705671,very large


In [15]:
feature_names = ['fireyear','lat','lon','month','temp_month','precip_month']

In [16]:
loadings = pca.components_.T
df_loadings = pd.DataFrame(loadings,columns = ['PC1','PC2','PC3'], index = feature_names)
df_loadings

,PC1,PC2,PC3
fireyear,-0.042276,0.657435,0.518390
lat,-0.348237,-0.190206,0.676193
lon,0.529235,0.189253,0.318572
month,-0.105126,0.571100,-0.399708
temp_month,-0.531299,0.358363,-0.110884
precip_month,0.550930,0.202989,-0.022036


In [17]:
explained_variance = pca.explained_variance_ratio_
explained_variance

array([0.27764088, 0.21329358, 0.16160244])

In [18]:
!pip install plotly


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [19]:
import numpy as np
import plotly.express as px

In [20]:
explained_variance = np.insert(explained_variance, 0, 0)

cumulative_variance = np.cumsum(np.round(explained_variance, decimals=3))

In [21]:
pc_df = pd.DataFrame(['','PC1', 'PC2', 'PC3'], columns=['PC'])
explained_variance_df = pd.DataFrame(explained_variance, columns=['Explained Variance'])
cumulative_variance_df = pd.DataFrame(cumulative_variance, columns=['Cumulative Variance'])

df_explained_variance = pd.concat([pc_df, explained_variance_df, cumulative_variance_df], axis=1)
df_explained_variance

,PC,Explained Variance,Cumulative Variance
0,,0.000000,0.000
1,PC1,0.277641,0.278
2,PC2,0.213294,0.491
3,PC3,0.161602,0.653
